# Predicting F1 Pit Stops — stable high-score notebook

Цель этой версии:
- запуск сверху вниз без ручных правок;
- отсутствие target leakage в OOF-валидации;
- безопасное добавление original dataset только в train-часть каждого фолда;
- стабильный ансамбль CatBoost + XGBoost + LightGBM;
- сохранение отдельных сабмитов и финального `submission.csv`.

`RealMLP / pytabkit` намеренно убран из основного пайплайна: он часто даёт CUDA OutOfMemory на Kaggle T4. Его можно добавить отдельно после получения стабильного baseline.

In [ ]:
# Optional installs for Kaggle, if some libraries are missing.
# Usually Kaggle already has xgboost/lightgbm/catboost installed.
# !pip install -q catboost lightgbm xgboost

In [ ]:
import os
import gc
import warnings
from itertools import combinations, product

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from scipy.stats import rankdata

from catboost import CatBoostClassifier, Pool
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

RANDOM_STATE = 42
N_SPLITS = 5
TARGET = "PitNextLap"
ID_COL = "id"
USE_ORIGINAL = True
USE_GPU = True  # если ловишь GPU/CUDA error, поставь False

In [ ]:
def seed_everything(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)

seed_everything(RANDOM_STATE)

## 1. Load data

Пути сделаны под Kaggle. Если название датасета отличается, поменяй только блок `PATHS`.

In [ ]:
PATHS = {
    "train": "/kaggle/input/competitions/playground-series-s6e5/train.csv",
    "test": "/kaggle/input/competitions/playground-series-s6e5/test.csv",
    "original": "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
}

def read_csv_safe(path: str, required: bool = True) -> pd.DataFrame | None:
    if os.path.exists(path):
        return pd.read_csv(path)
    if required:
        raise FileNotFoundError(f"File not found: {path}")
    print(f"Optional file not found: {path}")
    return None

train_raw = read_csv_safe(PATHS["train"], required=True)
test_raw = read_csv_safe(PATHS["test"], required=True)
orig_raw = read_csv_safe(PATHS["original"], required=False)

print("train:", train_raw.shape)
print("test :", test_raw.shape)
print("orig :", None if orig_raw is None else orig_raw.shape)

display(train_raw.head())

## 2. Cleaning and feature engineering

Все функции не меняют исходный dataframe in-place. Это важно для воспроизводимости.

In [ ]:
CAT_AS_OBJECT = ["Year", "Stint", "Position_Change"]
NGRAM_SOURCE_COLS = ["Compound", "Race", "Year"]
NUM_INTERACTION_COLS = ["LapNumber", "LapTime_Delta", "Cumulative_Degradation", "LapTime (s)", "TyreLife"]


def clean_base(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Убираем колонку из original, которой нет в competition train/test.
    if "Normalized_TyreLife" in df.columns:
        df = df.drop(columns=["Normalized_TyreLife"])

    # Приводим часть числовых колонок к категориям, как ты делал раньше.
    for col in CAT_AS_OBJECT:
        if col in df.columns:
            df[col] = df[col].astype("object")

    # Безопасная обработка явных выбросов. Не трогаем NaN.
    for col in ["LapTime (s)", "LapTime_Delta", "Cumulative_Degradation"]:
        if col in df.columns:
            med = df.loc[df[col].notna() & (df[col] <= 500), col].median()
            df.loc[df[col] > 500, col] = med

    return df


def add_bins(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "TyreLife" in df.columns:
        df["TyreLife_bin"] = pd.cut(df["TyreLife"], bins=[-1, 3, 7, 12, 18, 25, 10**9], labels=False).astype("object")
    if "RaceProgress" in df.columns:
        df["RaceProgress_bin"] = pd.cut(df["RaceProgress"], bins=np.linspace(0, 1, 11), labels=False, include_lowest=True).astype("object")
    if "LapNumber" in df.columns:
        df["LapNumber_bin"] = pd.cut(df["LapNumber"], bins=[-1, 5, 10, 15, 20, 30, 40, 60, 10**9], labels=False).astype("object")
    return df


def add_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    eps = 1e-6

    if {"LapNumber", "RaceProgress"}.issubset(df.columns):
        df["LapsRemaining_est"] = df["LapNumber"] * (1 - df["RaceProgress"]) / (df["RaceProgress"] + eps)

    if {"TyreLife", "RaceProgress"}.issubset(df.columns):
        df["TyreLife_to_RaceProgress"] = df["TyreLife"] / (df["RaceProgress"] + eps)

    if {"Stint", "TyreLife"}.issubset(df.columns):
        df["Stint_TyreLife_ratio"] = pd.to_numeric(df["Stint"], errors="coerce") / (df["TyreLife"] + 1)

    if "Position_Change" in df.columns:
        pos_ch = pd.to_numeric(df["Position_Change"], errors="coerce")
        df["Position_abs_change"] = pos_ch.abs()
        df["Is_Position_Lost"] = (pos_ch < 0).astype(int)
        df["Is_Position_Gained"] = (pos_ch > 0).astype(int)

    if "Cumulative_Degradation" in df.columns:
        df["High_Degradation"] = (df["Cumulative_Degradation"] > df["Cumulative_Degradation"].median()).astype(int)

    combo_pairs = [
        ("TyreLife", "Compound"),
        ("Race", "Compound"),
        ("Driver", "Compound"),
        ("Race", "Stint"),
        ("Driver", "Race"),
        ("Compound", "Stint"),
    ]
    for a, b in combo_pairs:
        if {a, b}.issubset(df.columns):
            df[f"{a}_x_{b}"] = df[a].astype(str) + "_" + df[b].astype(str)

    return df


def add_ngram_features(df: pd.DataFrame, source_cols=NGRAM_SOURCE_COLS, sizes=(2, 3)) -> pd.DataFrame:
    df = df.copy()
    source_cols = [c for c in source_cols if c in df.columns]
    new = {}
    for n in sizes:
        for cols in combinations(source_cols, n):
            new[f"{n}gram_" + "_".join(cols)] = df[list(cols)].astype(str).agg("_".join, axis=1)
    if new:
        df = pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)
    return df


def add_numeric_interactions(df: pd.DataFrame, cols=NUM_INTERACTION_COLS) -> pd.DataFrame:
    df = df.copy()
    cols = [c for c in cols if c in df.columns]
    eps = 1e-6
    new = {}
    for c1, c2 in combinations(cols, 2):
        s1 = pd.to_numeric(df[c1], errors="coerce")
        s2 = pd.to_numeric(df[c2], errors="coerce")
        base = f"{c1}_{c2}"
        new[f"{base}_mul"] = s1 * s2
        new[f"{base}_diff"] = s1 - s2
        new[f"{base}_sum"] = s1 + s2
        new[f"{base}_ratio"] = s1 / (s2.abs() + eps)
    if new:
        df = pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)
    return df


def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    df = clean_base(df)
    df = add_bins(df)
    df = add_domain_features(df)
    df = add_ngram_features(df)
    df = add_numeric_interactions(df)
    return df.replace([np.inf, -np.inf], np.nan)

In [ ]:
train = feature_engineering(train_raw)
test = feature_engineering(test_raw)
orig = feature_engineering(orig_raw) if orig_raw is not None else None

print("train FE:", train.shape)
print("test FE :", test.shape)
print("orig FE :", None if orig is None else orig.shape)

## 3. Safe column alignment

Original dataset может иметь другой набор колонок. Мы приводим его к competition train по признакам, иначе будут ошибки `out of bounds`, `feature mismatch`, `Cannot convert ... to float` и т.п.

In [ ]:
def split_xy(df: pd.DataFrame, target: str = TARGET, id_col: str = ID_COL):
    y = df[target].copy() if target in df.columns else None
    drop_cols = [c for c in [target, id_col] if c in df.columns]
    X = df.drop(columns=drop_cols)
    return X, y

X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET].astype(int)
X_test = test.drop(columns=[ID_COL])
test_ids = test[ID_COL].copy()

if orig is not None and TARGET in orig.columns:
    X_orig, y_orig = split_xy(orig)
    # Берём только колонки, которые есть в competition train. Недостающие добавляем NaN.
    X_orig = X_orig.reindex(columns=X.columns)
    y_orig = y_orig.astype(int)
else:
    X_orig, y_orig = None, None
    USE_ORIGINAL = False

# test тоже строго приводим к train columns
X_test = X_test.reindex(columns=X.columns)

print("X:", X.shape, "y:", y.shape)
print("X_test:", X_test.shape)
print("X_orig:", None if X_orig is None else X_orig.shape)

## 4. Fold-safe target/frequency encoding

Target encoding считается только на train-части текущего фолда. Validation не участвует в расчёте статистик, поэтому OOF честный.

In [ ]:
def get_cat_cols(df: pd.DataFrame) -> list[str]:
    return df.select_dtypes(include=["object", "string", "category"]).columns.tolist()


def get_num_cols(df: pd.DataFrame) -> list[str]:
    return df.select_dtypes(include=[np.number, "bool"]).columns.tolist()


def add_target_frequency_features(
    X_tr: pd.DataFrame,
    y_tr: pd.Series,
    X_val: pd.DataFrame,
    X_te: pd.DataFrame,
    cols: list[str],
    smoothing: float = 20.0,
):
    X_tr = X_tr.copy()
    X_val = X_val.copy()
    X_te = X_te.copy()

    global_mean = float(y_tr.mean())
    n_train = len(X_tr)
    work = X_tr[cols].copy()
    work[TARGET] = y_tr.values

    for col in cols:
        grp = work.groupby(col, dropna=False)[TARGET].agg(["mean", "count"])
        smooth_mean = (grp["mean"] * grp["count"] + global_mean * smoothing) / (grp["count"] + smoothing)
        freq = grp["count"] / n_train

        te_name = f"TE_{col}"
        fr_name = f"FE_{col}"

        for part in [X_tr, X_val, X_te]:
            part[te_name] = part[col].map(smooth_mean).fillna(global_mean).astype("float32")
            part[fr_name] = part[col].map(freq).fillna(0).astype("float32")

    return X_tr, X_val, X_te

## 5. Model-specific preprocessing

- CatBoost получает категориальные признаки как строки.
- XGBoost/LightGBM получают числовую матрицу: numeric impute + ordinal encoding категорий.

In [ ]:
def prepare_for_catboost(X_tr, X_val, X_te):
    X_tr = X_tr.copy()
    X_val = X_val.copy()
    X_te = X_te.copy()

    cat_cols = get_cat_cols(X_tr)
    num_cols = [c for c in X_tr.columns if c not in cat_cols]

    for col in cat_cols:
        for part in [X_tr, X_val, X_te]:
            part[col] = part[col].astype(str).fillna("missing")

    for col in num_cols:
        med = pd.to_numeric(X_tr[col], errors="coerce").median()
        for part in [X_tr, X_val, X_te]:
            part[col] = pd.to_numeric(part[col], errors="coerce").fillna(med).astype("float32")

    cat_idx = [X_tr.columns.get_loc(c) for c in cat_cols]
    return X_tr, X_val, X_te, cat_idx


def prepare_for_gbdt(X_tr, X_val, X_te):
    X_tr = X_tr.copy()
    X_val = X_val.copy()
    X_te = X_te.copy()

    cat_cols = get_cat_cols(X_tr)
    num_cols = [c for c in X_tr.columns if c not in cat_cols]

    if num_cols:
        imputer = SimpleImputer(strategy="median")
        X_tr[num_cols] = imputer.fit_transform(X_tr[num_cols])
        X_val[num_cols] = imputer.transform(X_val[num_cols])
        X_te[num_cols] = imputer.transform(X_te[num_cols])

    if cat_cols:
        for part in [X_tr, X_val, X_te]:
            part[cat_cols] = part[cat_cols].astype(str).fillna("missing")
        enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1)
        X_tr[cat_cols] = enc.fit_transform(X_tr[cat_cols])
        X_val[cat_cols] = enc.transform(X_val[cat_cols])
        X_te[cat_cols] = enc.transform(X_te[cat_cols])

    # Финальная защита от object dtype.
    X_tr = X_tr.astype("float32")
    X_val = X_val.astype("float32")
    X_te = X_te.astype("float32")
    return X_tr, X_val, X_te

## 6. Model configs

Параметры специально не экстремальные: сильнее стабильность, меньше риск OOM/таймаутов. Для финального пуша можно увеличить `n_estimators/iterations`.

In [ ]:
def make_models(use_gpu: bool = True):
    cat_params = dict(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=3500,
        learning_rate=0.025,
        depth=7,
        l2_leaf_reg=6,
        random_seed=RANDOM_STATE,
        auto_class_weights="Balanced",
        bootstrap_type="Bayesian",
        bagging_temperature=0.25,
        od_type="Iter",
        od_wait=250,
        verbose=300,
        allow_writing_files=False,
    )
    if use_gpu:
        cat_params.update(task_type="GPU")

    xgb_params = dict(
        n_estimators=3500,
        learning_rate=0.018,
        max_depth=5,
        min_child_weight=2,
        subsample=0.88,
        colsample_bytree=0.88,
        reg_alpha=0.08,
        reg_lambda=5.0,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    if use_gpu:
        # Для новых версий xgboost. Если версия старая, try/except ниже отловит.
        xgb_params.update(device="cuda")

    lgb_params = dict(
        objective="binary",
        metric="auc",
        n_estimators=5000,
        learning_rate=0.018,
        num_leaves=48,
        max_depth=-1,
        min_child_samples=30,
        subsample=0.88,
        subsample_freq=1,
        colsample_bytree=0.88,
        reg_alpha=0.05,
        reg_lambda=4.0,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
    )

    return {
        "cat": CatBoostClassifier(**cat_params),
        "xgb": XGBClassifier(**xgb_params),
        "lgb": LGBMClassifier(**lgb_params),
    }

## 7. Cross-validation training

Original dataset добавляется только к `X_tr/y_tr` внутри каждого фолда. Validation остаётся чистой частью competition train.

In [ ]:
def fit_predict_one_model(model_name, base_model, X, y, X_test, X_orig=None, y_orig=None):
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(X), dtype=np.float32)
    test_pred = np.zeros(len(X_test), dtype=np.float32)
    fold_scores = []

    cat_cols_base = get_cat_cols(X)
    te_cols = [
        c for c in cat_cols_base
        if X[c].nunique(dropna=False) <= 300
    ]
    # Добавляем несколько числовых дискретных признаков, если они есть.
    for c in ["LapNumber", "TyreLife", "RaceProgress_bin", "TyreLife_bin", "LapNumber_bin"]:
        if c in X.columns and c not in te_cols:
            te_cols.append(c)

    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y), 1):
        print("=" * 90)
        print(f"{model_name.upper()} | fold {fold}/{N_SPLITS}")

        X_tr = X.iloc[tr_idx].copy()
        y_tr = y.iloc[tr_idx].copy()
        X_val = X.iloc[val_idx].copy()
        y_val = y.iloc[val_idx].copy()
        X_te = X_test.copy()

        # Safe original augmentation: только в train fold.
        if USE_ORIGINAL and X_orig is not None and y_orig is not None:
            X_tr = pd.concat([X_tr, X_orig.copy()], axis=0, ignore_index=True)
            y_tr = pd.concat([y_tr.reset_index(drop=True), y_orig.reset_index(drop=True)], axis=0, ignore_index=True)

        X_tr, X_val, X_te = add_target_frequency_features(X_tr, y_tr, X_val, X_te, cols=te_cols, smoothing=25.0)

        model = clone(base_model)

        if model_name == "cat":
            X_tr_p, X_val_p, X_te_p, cat_idx = prepare_for_catboost(X_tr, X_val, X_te)
            try:
                model.fit(
                    X_tr_p, y_tr,
                    eval_set=(X_val_p, y_val),
                    cat_features=cat_idx,
                    use_best_model=True,
                )
            except Exception as e:
                if USE_GPU:
                    print("CatBoost GPU failed, fallback to CPU. Error:", repr(e))
                    params = model.get_params()
                    params.pop("task_type", None)
                    model = CatBoostClassifier(**params)
                    model.fit(
                        X_tr_p, y_tr,
                        eval_set=(X_val_p, y_val),
                        cat_features=cat_idx,
                        use_best_model=True,
                    )
                else:
                    raise
            val_pred = model.predict_proba(X_val_p)[:, 1]
            te_pred = model.predict_proba(X_te_p)[:, 1]

        else:
            X_tr_p, X_val_p, X_te_p = prepare_for_gbdt(X_tr, X_val, X_te)
            fit_kwargs = {}
            if model_name == "xgb":
                fit_kwargs = {"eval_set": [(X_val_p, y_val)], "verbose": 300}
            elif model_name == "lgb":
                fit_kwargs = {"eval_set": [(X_val_p, y_val)]}

            try:
                model.fit(X_tr_p, y_tr, **fit_kwargs)
            except TypeError:
                # На случай старой версии библиотеки, которая не принимает часть kwargs.
                model.fit(X_tr_p, y_tr)
            except Exception as e:
                if model_name == "xgb" and USE_GPU:
                    print("XGBoost GPU failed, fallback to CPU. Error:", repr(e))
                    params = model.get_params()
                    params.pop("device", None)
                    model = XGBClassifier(**params)
                    model.fit(X_tr_p, y_tr, eval_set=[(X_val_p, y_val)], verbose=300)
                else:
                    raise

            val_pred = model.predict_proba(X_val_p)[:, 1]
            te_pred = model.predict_proba(X_te_p)[:, 1]

        fold_auc = roc_auc_score(y_val, val_pred)
        fold_scores.append(fold_auc)
        oof[val_idx] = val_pred.astype(np.float32)
        test_pred += te_pred.astype(np.float32) / N_SPLITS

        print(f"Fold {fold} AUC: {fold_auc:.6f}")
        del X_tr, X_val, X_te, X_tr_p, X_val_p, X_te_p, model
        gc.collect()

    full_auc = roc_auc_score(y, oof)
    print("=" * 90)
    print(f"{model_name.upper()} FULL OOF AUC: {full_auc:.6f}")
    print(f"{model_name.upper()} fold mean/std: {np.mean(fold_scores):.6f} +/- {np.std(fold_scores):.6f}")
    return oof, test_pred, fold_scores

In [ ]:
models = make_models(USE_GPU)

all_oof = {}
all_test_preds = {}
all_fold_scores = {}

# Для быстрого теста можешь оставить только ["cat"]. Для финала запускай все три.
RUN_MODELS = ["cat", "xgb", "lgb"]

for name in RUN_MODELS:
    oof_pred, test_pred, scores = fit_predict_one_model(
        name,
        models[name],
        X,
        y,
        X_test,
        X_orig=X_orig,
        y_orig=y_orig,
    )
    all_oof[name] = oof_pred
    all_test_preds[name] = test_pred
    all_fold_scores[name] = scores

summary = pd.DataFrame({
    "model": list(all_oof.keys()),
    "oof_auc": [roc_auc_score(y, all_oof[m]) for m in all_oof],
    "fold_mean": [np.mean(all_fold_scores[m]) for m in all_fold_scores],
    "fold_std": [np.std(all_fold_scores[m]) for m in all_fold_scores],
}).sort_values("oof_auc", ascending=False)

display(summary)

## 8. Save single-model submissions

In [ ]:
for model_name, pred in all_test_preds.items():
    sub = pd.DataFrame({ID_COL: test_ids, TARGET: pred})
    path = f"submission_{model_name}.csv"
    sub.to_csv(path, index=False)
    print("saved", path)

## 9. OOF-weighted blending

Здесь веса подбираются по OOF. Для 3 моделей используется небольшой grid search по весам. Дополнительно есть rank blend — часто он даёт более устойчивый public score.

In [ ]:
def find_best_linear_blend(oofs: dict[str, np.ndarray], y_true: pd.Series, step: float = 0.02):
    names = list(oofs.keys())
    if len(names) == 1:
        return {names[0]: 1.0}, roc_auc_score(y_true, oofs[names[0]])

    best_auc = -np.inf
    best_weights = None

    if len(names) == 2:
        grid = np.arange(0, 1 + 1e-9, step)
        for w in grid:
            weights = np.array([w, 1 - w])
            pred = sum(weights[i] * oofs[names[i]] for i in range(2))
            auc = roc_auc_score(y_true, pred)
            if auc > best_auc:
                best_auc = auc
                best_weights = weights
    else:
        grid = np.arange(0, 1 + 1e-9, step)
        for ws in product(grid, repeat=len(names)):
            s = sum(ws)
            if abs(s - 1.0) > 1e-9:
                continue
            weights = np.array(ws)
            pred = sum(weights[i] * oofs[names[i]] for i in range(len(names)))
            auc = roc_auc_score(y_true, pred)
            if auc > best_auc:
                best_auc = auc
                best_weights = weights

    return dict(zip(names, best_weights)), best_auc

best_weights, best_blend_auc = find_best_linear_blend(all_oof, y, step=0.02)
print("Best OOF weights:", best_weights)
print("Best linear blend OOF AUC:", best_blend_auc)

linear_test = np.zeros(len(X_test), dtype=np.float32)
linear_oof = np.zeros(len(X), dtype=np.float32)
for name, w in best_weights.items():
    linear_test += w * all_test_preds[name]
    linear_oof += w * all_oof[name]

# Rank blend with the same weights. It is less sensitive to calibration differences.
rank_test = np.zeros(len(X_test), dtype=np.float32)
rank_oof = np.zeros(len(X), dtype=np.float32)
for name, w in best_weights.items():
    rank_test += w * (rankdata(all_test_preds[name]) / len(X_test))
    rank_oof += w * (rankdata(all_oof[name]) / len(X))

print("Linear OOF AUC:", roc_auc_score(y, linear_oof))
print("Rank OOF AUC  :", roc_auc_score(y, rank_oof))

In [ ]:
submission_linear = pd.DataFrame({ID_COL: test_ids, TARGET: linear_test})
submission_rank = pd.DataFrame({ID_COL: test_ids, TARGET: rank_test})

submission_linear.to_csv("submission_linear_blend.csv", index=False)
submission_rank.to_csv("submission_rank_blend.csv", index=False)

# Финальный файл для отправки. По умолчанию rank blend, потому что он часто стабильнее на leaderboard.
submission = submission_rank.copy()
submission.to_csv("submission.csv", index=False)

print("Saved: submission.csv, submission_linear_blend.csv, submission_rank_blend.csv")
display(submission.head())

## 10. Notes for further improvement

1. Если ноутбук проходит стабильно — увеличь `iterations/n_estimators` на 20–40%.
2. Если ловишь OOM — `USE_GPU = False` или запускай модели по одной: `RUN_MODELS = ["cat"]`, потом `["xgb"]`, потом `["lgb"]`.
3. Не добавляй original dataset в validation. Это сломает честность OOF.
4. Не делай feature selection через модель внутри каждого фолда без строгого контроля: часто результат хуже и код становится нестабильным.
5. `pytabkit/RealMLP` можно добавить отдельным экспериментом, но не в основной notebook: он слишком часто падает по памяти.